# Interaction Analysis: Monastic Land and Nearby Rebellion (Contagion)

This notebook tests whether the presence of monastic land amplifies the spatial spillover effect from nearby rebelling muster points.

Hypothesis: Parishes with high monastic land intensity are more susceptible to "infection" from neighboring rebellion.

In [1]:
pacman::p_load(
  sf, tidyverse, stargazer, dplyr,
  spdep, ggplot2, conleyreg, jsonlite
)

PROJECT_ROOT <- ".."
setwd(PROJECT_ROOT)
pretty_dict <- fromJSON("Code/pretty_dict.json")

pdf <- read_sf(dsn = "Data/Processed/northParishFlows.shp")

# near_reb_m is already binary (0/1)
# Standardize continuous monastic variables
continuous_vars <- c("lbg_arak", "loth_arak", "lni_arak", "lpopC", "lLStax_pc")
for (v in continuous_vars) {
  pdf[[v]] <- scale(pdf[[v]], center = TRUE, scale = TRUE)[, 1]
}

hide_vars <- c("area", "mean_slope", "uplands", "lowlands", "distScot", "Constant")
controls_formula <- "lpopC + lLStax_pc + area + mean_slope + uplands + lowlands + distScot"


## Section 1: Interaction with Nearby Muster Points

Testing `monastic_var * near_reb_m`

In [2]:
models_contagion <- list(
  c1 = glm(paste0("primary ~ lbg_arak * near_reb_m + ", controls_formula), data = pdf, family = binomial(link = "logit")),
  c2 = glm(paste0("primary ~ loth_arak * near_reb_m + ", controls_formula), data = pdf, family = binomial(link = "logit")),
  c3 = glm(paste0("primary ~ lni_arak * near_reb_m + ", controls_formula), data = pdf, family = binomial(link = "logit"))
)

stargazer(models_contagion, type = "text", 
          dep.var.labels = "Primary",
          column.labels = c("Large House", "Off-site", "Net Income"),
          omit = hide_vars)


                                                              Dependent variable:                                          
                     ------------------------------------------------------------------------------------------------------
                     primary ~ lbg_arak * near_reb_m + primary ~ loth_arak * near_reb_m + primary ~ lni_arak * near_reb_m +
                                Large House                         Off-site                         Net Income            
                                    (1)                               (2)                                (3)               
---------------------------------------------------------------------------------------------------------------------------
lbg_arak                           0.355                                                                                   
                                  (1.398)                                                                                  
       